In [36]:
#1
import pandas as pd
import numpy as np
import re
from pathlib import Path


DATA = Path("/content/drive/MyDrive/dirisa-sdc/raw")
PROCESSED = Path("/content/drive/MyDrive/dirisa-sdc/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)


def clean_percent(series):
    """Convert '45.6%' -> 45.6"""
    return (
        series.astype(str)
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan})
        .astype(float)
    )

def clean_number(series):
    """Convert '1 234' or '1,234' -> 1234"""
    return (
        series.astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan})
        .astype(float)
    )

def extract_code(name):
    """Extract IEC mdb_code from 'EC101 - Dr. Beyers Naude' -> 'EC101'"""
    if pd.isna(name):
        return np.nan
    match = re.match(r"^([A-Z]{2,3}\d*)\s*-", str(name).strip())
    return match.group(1) if match else str(name).strip()

In [37]:

#2
age = pd.read_csv(DATA / "ec_age_gender (1).csv")

print("Raw shape:", age.shape)
print("Columns:", age.columns.tolist())
print("\nUnique municipalities:", age["municipality"].nunique())
print("Sample municipalities:")
print(age["municipality"].drop_duplicates().head(10).tolist())


age.columns = [c.strip().lower().replace(" ", "_") for c in age.columns]


age = age[age["age_group"] != "Overall"].copy()

age["total"]   = clean_number(age["total"])
age["percent"] = clean_percent(age["percent"])


age["mdb_code"] = age["municipality"].apply(extract_code)
age["province"] = age["province"].str.strip()


age = age.dropna(subset=["total"]).copy()

print("\nCleaned age_gender shape:", age.shape)
print("Unique mdb_codes:", age["mdb_code"].nunique())
print("Age groups present:", sorted(age["age_group"].unique()))

Raw shape: (594, 6)
Columns: ['age_group', 'gender', 'total', 'percent', 'province', 'municipality']

Unique municipalities: 33
Sample municipalities:
['BUF - Buffalo City', 'EC101 - Dr. Beyers Naude', 'EC102 - Blue Crane Route', 'EC104 - Makana', 'EC105 - Ndlambe', 'EC106 - Sundays River Valley', 'EC108 - Kouga', 'EC109 - Kou-Kamma', 'EC121 - Mbhashe', 'EC122 - Mnquma']

Cleaned age_gender shape: (528, 7)
Unique mdb_codes: 33
Age groups present: ['18', '20', '30', '40', '50', '60', '70', '80']


In [38]:

#3
youth_bands = ["18", "20", "30"]

age["age_band_start"] = age["age_group"].astype(str).str.strip()

youth_reg = (
    age[age["age_band_start"].isin(youth_bands)]
    .groupby(["province", "mdb_code", "municipality"], as_index=False)
    .agg(registered_youth_total=("total", "sum"))
)

print("Youth registration by municipality:")
print(youth_reg.head(10))
print("\nTotal municipalities with youth data:", len(youth_reg))

Youth registration by municipality:
       province mdb_code                  municipality  registered_youth_total
0  Eastern Cape      BUF            BUF - Buffalo City                177827.0
1  Eastern Cape    EC101      EC101 - Dr. Beyers Naude                 17751.0
2  Eastern Cape    EC102      EC102 - Blue Crane Route                  7445.0
3  Eastern Cape    EC104                EC104 - Makana                 19589.0
4  Eastern Cape    EC105               EC105 - Ndlambe                 12919.0
5  Eastern Cape    EC106  EC106 - Sundays River Valley                 10763.0
6  Eastern Cape    EC108                 EC108 - Kouga                 24542.0
7  Eastern Cape    EC109             EC109 - Kou-Kamma                 10659.0
8  Eastern Cape    EC121               EC121 - Mbhashe                 54265.0
9  Eastern Cape    EC122                EC122 - Mnquma                 48601.0

Total municipalities with youth data: 33


In [39]:
#4

master = pd.read_csv(DATA / "ec_master_municipality_level.csv")

print("Master shape:", master.shape)
print("Columns:", master.columns.tolist())
print("\nSample:")
print(master.head(3))


master.columns = [c.strip().lower().replace(" ", "_") for c in master.columns]


num_cols = [
    "total_pop_2022", "youth_pop_20_34_2022", "youth_share_pct_2022",
    "registered_2016", "valid_votes_2016", "spoilt_votes_2016",
    "votes_cast_2016", "turnout_pct_2016", "spoilt_rate_pct_2016",
    "registered_2021", "valid_votes_2021", "spoilt_votes_2021",
    "votes_cast_2021", "turnout_pct_2021", "spoilt_rate_pct_2021",
    "registered_growth_2016_2021_pct", "turnout_change_2016_2021_pp",
]
for c in num_cols:
    if c in master.columns:
        master[c] = clean_number(master[c])


master["mdb_code"] = master["mdb_code"].astype(str).str.strip()
master["municipality"] = master["municipality"].astype(str).str.strip()

print("\nMaster cleaned. Rows:", len(master))
print("Missing 2021 turnout rows:",
      master["turnout_pct_2021"].isna().sum())

Master shape: (33, 20)
Columns: ['mdb_code', 'municipality', 'district', 'total_pop_2022', 'youth_pop_20_34_2022', 'youth_share_pct_2022', 'registered_2016', 'valid_votes_2016', 'spoilt_votes_2016', 'votes_cast_2016', 'turnout_pct_2016', 'spoilt_rate_pct_2016', 'registered_2021', 'valid_votes_2021', 'spoilt_votes_2021', 'votes_cast_2021', 'turnout_pct_2021', 'spoilt_rate_pct_2021', 'registered_growth_2016_2021_pct', 'turnout_change_2016_2021_pp']

Sample:
  mdb_code      municipality        district  total_pop_2022  \
0    EC124         Amahlathi        Amathole        115703.0   
1    EC102  Blue Crane Route  Sarah Baartman         49883.0   
2      BUF      Buffalo City    Buffalo City        975249.0   

   youth_pop_20_34_2022  youth_share_pct_2022  registered_2016  \
0               24608.0             21.268247            52945   
1               11309.0             22.671050            18372   
2              238729.0             24.478774           419044   

   valid_votes_201

In [40]:


#5
district_youth = pd.read_csv(DATA / "District MunicipalitiesEC.csv")

print("District youth shape:", district_youth.shape)
print("Columns:", district_youth.columns.tolist())

district_youth.columns = [c.strip().lower().replace(" ", "_") for c in district_youth.columns]


age_cols = ["18-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
for c in age_cols:
    if c in district_youth.columns:
        district_youth[c] = clean_number(district_youth[c])

district_youth["registered_youth_total"] = (
    district_youth["18-19"] +
    district_youth["20-29"] +
    district_youth["30-39"]
)

print("\nDistrict youth with computed totals:")
print(district_youth[["district_municipality", "year", "registered_youth_total"]].head(10))

District youth shape: (12, 10)
Columns: ['District Municipality', '18-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80+', 'Year']

District youth with computed totals:
            district_municipality  year  registered_youth_total
0             Alfred Nzo District  2021                198620.0
1             Alfred Nzo District  2016                205307.0
2  Amathole District Municipality  2021                166792.0
3  Amathole District Municipality  2016                189825.0
4                      Chris Hani  2021                 39271.0
5                      Chris Hani  2016                 43607.0
6              Joe Gqabi District  2021                 86332.0
7              Joe Gqabi District  2016                100150.0
8               OR Tambo District  2021                331206.0
9               OR Tambo District  2016                367948.0


In [41]:
#6

pop = pd.read_csv(DATA / "TotalPopulationEc.csv")

print("Population shape:", pop.shape)
pop.columns = [c.strip().lower().replace(" ", "_") for c in pop.columns]


pop = pop.rename(columns={
    "district_municipality": "district",
    "0-9": "pop_0_9", "10-19": "pop_10_19", "20-29": "pop_20_29",
    "30-39": "pop_30_39", "40-49": "pop_40_49", "50-59": "pop_50_59",
    "60-69": "pop_60_69", "70-79": "pop_70_79",
    "80-89": "pop_80_89", "90-99": "pop_90_99",
})

for c in [x for x in pop.columns if x.startswith("pop_")]:
    pop[c] = clean_number(pop[c])

pop["eligible_youth_15_34_approx"] = (
    pop["pop_10_19"] * 0.5 + pop["pop_20_29"] + pop["pop_30_39"] * 0.5
)

print("\nPopulation with youth approximation:")
print(pop[["district", "eligible_youth_15_34_approx"]])

Population shape: (6, 15)

Population with youth approximation:
                               district  eligible_youth_15_34_approx
0       Joe Gqabi District Municipality                     124887.0
1  Sarah Baartman District Municipality                     166151.0
2      Chris Hani District Municipality                     254103.5
3        Amathole District Municipality                     265141.5
4      Alfred Nzo District Municipality                     320544.0
5        OR Tambo District Municipality                     519951.5


In [42]:
#7

turnout = pd.read_csv(DATA / "Percentage_Voter_Turnout_2021.csv", skiprows=10, sep=";", engine="python")


turnout = turnout.dropna(axis=1, how="all").dropna(how="all")
turnout.columns = [str(c).strip() for c in turnout.columns]

print("Turnout shape:", turnout.shape)
print("Columns:", turnout.columns.tolist())
print(turnout.head())

Turnout shape: (13, 5)
Columns: ['Unnamed: 1', 'Unnamed: 6', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 12']
                                          Unnamed: 1           Unnamed: 6  \
0  All information in the report pertains to audi...                  NaN   
2  % Voter Turnout = (Highest of either Ward and ...                  NaN   
5                                           Province  Registered \nVoters   
6                                       Eastern Cape            3 253 307   
7                                         Free State            1 413 158   

    Unnamed: 8      Unnamed: 9        Unnamed: 12  
0          NaN             NaN                NaN  
2          NaN             NaN                NaN  
5  MEC7\nVotes  Voter\nTurnout  % Voter \nTurnout  
6       17 923       1 522 726             46,55%  
7        5 643         643 417             45,35%  


In [43]:
#8
youth_reg.to_csv(PROCESSED / "clean_youth_registration.csv", index=False)


master.to_csv(PROCESSED / "clean_master_municipality.csv", index=False)


district_youth.to_csv(PROCESSED / "clean_district_youth_history.csv", index=False)


pop.to_csv(PROCESSED / "clean_census_population.csv", index=False)


turnout.to_csv(PROCESSED / "clean_provincial_turnout_2021.csv", index=False)

print("✅ All cleaned files written to:", PROCESSED)
for f in PROCESSED.glob("clean_*.csv"):
    print(f"   {f.name}")

✅ All cleaned files written to: /content/drive/MyDrive/dirisa-sdc/processed
   clean_youth_registration.csv
   clean_master_municipality.csv
   clean_district_youth_history.csv
   clean_census_population.csv
   clean_provincial_turnout_2021.csv


In [44]:
#9

print("\n=== Data quality summary ===")
print(f"Municipalities in master:          {master['mdb_code'].nunique()}")
print(f"Municipalities in youth_reg:       {youth_reg['mdb_code'].nunique()}")

overlap = set(master["mdb_code"]) & set(youth_reg["mdb_code"])
print(f"Municipalities in BOTH:            {len(overlap)}")

print("\nIn master but NOT in youth_reg:")
print(sorted(set(master["mdb_code"]) - set(youth_reg["mdb_code"])))

print("\nIn youth_reg but NOT in master:")
print(sorted(set(youth_reg["mdb_code"]) - set(master["mdb_code"])))


=== Data quality summary ===
Municipalities in master:          33
Municipalities in youth_reg:       33
Municipalities in BOTH:            33

In master but NOT in youth_reg:
[]

In youth_reg but NOT in master:
[]


In [45]:
import pandas as pd
youth = pd.read_csv("/content/processed/clean_youth_registration.csv")
print("Shape:", youth.shape)
print("Columns:", youth.columns.tolist())
print()
print(youth.head(5))

Shape: (33, 4)
Columns: ['province', 'mdb_code', 'municipality', 'registered_youth_total']

       province mdb_code              municipality  registered_youth_total
0  Eastern Cape      BUF        BUF - Buffalo City                177827.0
1  Eastern Cape    EC101  EC101 - Dr. Beyers Naude                 17751.0
2  Eastern Cape    EC102  EC102 - Blue Crane Route                  7445.0
3  Eastern Cape    EC104            EC104 - Makana                 19589.0
4  Eastern Cape    EC105           EC105 - Ndlambe                 12919.0


In [46]:
import pandas as pd
district = pd.read_csv("/content/processed/clean_district_youth_history.csv")
print("Shape:", district.shape)
print("Columns:", district.columns.tolist())
print()
print(district.head(6))

Shape: (12, 11)
Columns: ['district_municipality', '18-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80+', 'year', 'registered_youth_total']

            district_municipality    18-19     20-29    30-39    40-49  \
0             Alfred Nzo District   3511.0   97421.0  97688.0  69032.0   
1             Alfred Nzo District  14472.0  107280.0  83555.0  58579.0   
2  Amathole District Municipality   1934.0   72795.0  92063.0  75079.0   
3  Amathole District Municipality   9481.0   94992.0  85352.0  72352.0   
4                      Chris Hani    651.0   15957.0  22663.0  18303.0   
5                      Chris Hani   2064.0   20878.0  20665.0  17871.0   

     50-59    60-69    70-79      80+  year  registered_youth_total  
0  51809.0  42626.0  24468.0  19859.0  2021                198620.0  
1  51986.0  38941.0  23606.0  18527.0  2016                205307.0  
2  68443.0  60140.0  32552.0  22678.0  2021                166792.0  
3  73666.0  56800.0  34784.0  21227.0  2016  

In [34]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
from pathlib import Path
Path("/content/drive/MyDrive/dirisa-sdc/raw").mkdir(parents=True, exist_ok=True)
Path("/content/drive/MyDrive/dirisa-sdc/processed").mkdir(parents=True, exist_ok=True)
print("Folders created.")

Folders created.
